# Credit Card Transaction Analysis & Fraud Detection
## Stage 3: Machine Learning Modeling & Comparison

In this notebook, we train, evaluate, and compare two supervised learning models:
1. **Logistic Regression (Class-Weighted Baseline)**
2. **Random Forest Classifier (Ensemble Method)**

Due to extreme class imbalance (less than 1% fraud), we optimize for Recall, Precision, F1-Score, and ROC-AUC rather than standard accuracy.

### Step 1: Load Cleaned Datasets & Select Features

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

train = pd.read_csv(os.path.join("..", "data", "processed", "fraudTrain_clean.csv"))
test = pd.read_csv(os.path.join("..", "data", "processed", "fraudTest_clean.csv"))

features = ["amt", "category", "gender", "city_pop", "transaction_hour", "day_of_week", "is_weekend", "customer_age", "merchant_distance"]
target = "is_fraud"

# Take a stratified 200k subset for fast iteration
train_sample = train.groupby(target, group_keys=False).apply(lambda x: x.sample(min(len(x), int(200000 * len(x) / len(train))), random_state=42))
test_sample = test.groupby(target, group_keys=False).apply(lambda x: x.sample(min(len(x), int(50000 * len(x) / len(test))), random_state=42))

X_train, y_train = train_sample[features], train_sample[target]
X_test, y_test = test_sample[features], test_sample[target]
print(f"Train sample: {X_train.shape}, Test sample: {X_test.shape}")

### Step 2: Build Preprocessing Pipeline

In [ ]:
numeric_features = ["amt", "city_pop", "transaction_hour", "day_of_week", "is_weekend", "customer_age", "merchant_distance"]
categorical_features = ["category", "gender"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

### Step 3: Model 1 — Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=["Genuine", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

### Step 4: Model 2 — Random Forest Classifier

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=14, class_weight="balanced", random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["Genuine", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

### Step 5: ROC Curve Comparison

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.title("ROC Curves Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()